 # Exploração Segura de Dados da Receita Federal com DuckDB

 Executa consultas analíticas diretamente nos arquivos Parquet com baixo consumo de RAM.

In [ ]:
import os
from pathlib import Path
!poetry add duckdb --group notebooks
import duckdb

# Inicializar conexão in-memory
con = duckdb.connect(database=":memory:")

# Limitar o uso de RAM para proteger o sistema operacional
# Ajuste conforme sua folga (ex: '4GB' ou '6GB' garante que o OOM não atue)
con.execute("SET max_memory = '4GB';")
con.execute("SET preserve_insertion_order = false;")

print("DuckDB configurado com limite seguro de memória.")


 ### 1. Verificação dos Arquivos Gerados

In [ ]:
# Ancoragem robusta: pega o diretório pai da pasta 'notebooks' (ou seja, a raiz do repo)
ROOT_DIR = Path(__file__).resolve().parent.parent if "__file__" in locals() else Path.cwd().parent
PASTA_PARQUET = ROOT_DIR / "tmp_receita"
MES_TESTE = "2024-01"

path_empresa = str(PASTA_PARQUET / f"empresa_{MES_TESTE}.parquet")
path_estabele = str(PASTA_PARQUET / f"estabele_{MES_TESTE}.parquet")
path_socio = str(PASTA_PARQUET / f"socio_{MES_TESTE}.parquet")
path_simples = str(PASTA_PARQUET / f"simples_{MES_TESTE}.parquet")

print(f"Diretório raiz detectado: {ROOT_DIR}")
print(f"Buscando arquivos em: {PASTA_PARQUET}")
print("-" * 50)

for p in [path_empresa, path_estabele, path_socio, path_simples]:
    status = "✅ Encontrado" if os.path.exists(p) else "❌ Não encontrado"
    print(f"{status}: {os.path.basename(p)}")


 ### 2. Metadados e Contagem Total de Linhas

 O DuckDB lê apenas os cabeçalhos Parquet para contar linhas quase instantaneamente.

In [ ]:
if os.path.exists(path_empresa):
    query_count = f"""
    SELECT 
        count(*) as total_linhas,
        count(DISTINCT cnpj_basico) as cnpjs_unicos
    FROM read_parquet('{path_empresa}')
    """
    resumo_empresa = con.execute(query_count).df()
    print("Métricas da Tabela EMPRESA:")
    print(resumo_empresa)


 ### 3. Validação do Mascaramento do MEI

 Consulta registros para confirmar se o CPF foi substituído pelo CNPJ básico na Razão Social.

In [ ]:
if os.path.exists(path_empresa):
    print("--- TESTE 1: MEIs mascarados com sucesso (terminam com exatos 8 números) ---")
    query_mei_sucesso = f"""
    SELECT 
        cnpj_basico,
        razao_social
    FROM read_parquet('{path_empresa}')
    WHERE porte_empresa = '01'
      AND natureza_juridica = '2135' 
      AND razao_social ~ ' [0-9]{{8}}$' -- Espaço seguido de exatamente 8 dígitos no final
    LIMIT 5
    """
    print(con.execute(query_mei_sucesso).df())

    print("\n--- TESTE 2: MEIs que vazaram (ainda possuem 11 números do CPF) ---")
    query_cpf_vazado = f"""
    SELECT 
        cnpj_basico,
        razao_social
    FROM read_parquet('{path_empresa}')
    WHERE porte_empresa = '01'
      AND natureza_juridica = '2135'
      -- Busca CPF com ou sem formatação (ex: 12345678900 ou 123.456.789-00)
      AND regexp_matches(razao_social, '[0-9]{{3}}\\.?[0-9]{{3}}\\.?[0-9]{{3}}-?[0-9]{{2}}')
    LIMIT 20
    """
    df_vazados = con.execute(query_cpf_vazado).df()
    print("CPFs que sobreviveram na base (antes da sua máscara atuar):")
    print(df_vazados)


In [ ]:
if os.path.exists(path_empresa):
    query_investigacao = f"""
    SELECT 
        cnpj_basico,
        razao_social
    FROM read_parquet('{path_empresa}')
    WHERE porte_empresa = '01'
      AND natureza_juridica = '2135'
    LIMIT 10
    """
    df_investigacao = con.execute(query_investigacao).df()
    print("Formato real da Razão Social do MEI:")
    print(df_investigacao)

 ### 4. Distribuição por Porte de Empresa

 Agregação pesada executada via DuckDB sem estourar memória.

In [ ]:
if os.path.exists(path_empresa):
    query_porte = f"""
    SELECT 
        porte_empresa,
        count(*) as total_empresas,
        round(count(*) * 100.0 / sum(count(*)) over (), 2) as percentual
    FROM read_parquet('{path_empresa}')
    GROUP BY porte_empresa
    ORDER BY total_empresas DESC
    """
    df_porte = con.execute(query_porte).df()
    print("Distribuição por Porte:")
    print(df_porte)


 ### 5. Amostra de Estabelecimentos Ativos

 Leitura com projeção de colunas específicas (evita ler as 30 colunas do disco).

In [ ]:
if os.path.exists(path_estabele):
    query_estab = f"""
    SELECT 
        cnpj_basico,
        cnpj_ordem,
        cnpj_dv,
        nome_fantasia,
        situacao_cadastral,
        uf,
        municipio,
        data_inicio_atividade
    FROM read_parquet('{path_estabele}')
    WHERE situacao_cadastral = '02' -- Ativa
      AND uf IS NOT NULL
    LIMIT 10
    """
    amostra_estab = con.execute(query_estab).df()
    print("Amostra Estabelecimentos Ativos:")
    print(amostra_estab)


 ### 6. Cruzamento (JOIN) Seguro em Disco

 Junta Empresa e Estabelecimento trazendo apenas o topo por estado.

In [ ]:
if os.path.exists(path_empresa) and os.path.exists(path_estabele):
    query_join = f"""
    SELECT 
        e.uf,
        count(*) as total_matrizes_ativas
    FROM read_parquet('{path_estabele}') e
    JOIN read_parquet('{path_empresa}') emp ON e.cnpj_basico = emp.cnpj_basico
    WHERE e.identificador_matriz_filial = '1' -- Matriz
      AND e.situacao_cadastral = '02'        -- Ativa
      AND e.uf != ''
    GROUP BY e.uf
    ORDER BY total_matrizes_ativas DESC
    LIMIT 10
    """
    top_ufs = con.execute(query_join).df()
    print("Top 10 UFs com Matrizes Ativas:")
    print(top_ufs)



 ### 7. Explorando as Tabelas de Domínio

 Verificando a carga dos Municípios e CNAEs (Atividades Econômicas).

In [ ]:
path_municipios = str(PASTA_PARQUET / f"municipios_{MES_TESTE}.parquet")
path_cnaes = str(PASTA_PARQUET / f"cnaes_{MES_TESTE}.parquet")

if os.path.exists(path_municipios):
    query_mun = f"""
    SELECT codigo, descricao
    FROM read_parquet('{path_municipios}')
    LIMIT 5
    """
    df_mun = con.execute(query_mun).df()
    print("Amostra Tabela MUNICÍPIOS:")
    print(df_mun)

if os.path.exists(path_cnaes):
    query_cnae = f"""
    SELECT codigo, descricao
    FROM read_parquet('{path_cnaes}')
    LIMIT 5
    """
    df_cnae = con.execute(query_cnae).df()
    print("\nAmostra Tabela CNAES:")
    print(df_cnae)


 ### 8. JOIN Completo: Estabelecimentos + Municípios + CNAEs

 Trazendo o nome real do município e a descrição da atividade principal do estabelecimento em tempo recorde.

In [ ]:
if os.path.exists(path_estabele) and os.path.exists(path_municipios) and os.path.exists(path_cnaes):
    query_completa = f"""
    SELECT 
        e.cnpj_basico || e.cnpj_ordem || e.cnpj_dv AS cnpj_completo,
        e.nome_fantasia,
        m.descricao AS nome_municipio,
        e.uf,
        c.descricao AS atividade_principal
    FROM read_parquet('{path_estabele}') e
    LEFT JOIN read_parquet('{path_municipios}') m ON e.municipio = m.codigo
    LEFT JOIN read_parquet('{path_cnaes}') c ON e.cnae_fiscal_principal = c.codigo
    WHERE e.situacao_cadastral = '02' -- Ativa
      AND e.uf = 'RJ'                 -- Apenas Rio de Janeiro
    LIMIT 10
    """
    df_completo = con.execute(query_completa).df()
    print("Estabelecimentos Enriquecidos (JOIN com Domínios):")
    print(df_completo)

# Testar mascaramento baixando da fonte

In [ ]:
import os
import re
import zipfile
import requests
import pandas as pd

TOKEN_SHARE = "YggdBLfdninEJX9"
URL_ZIP = f"https://arquivos.receitafederal.gov.br/public.php/dav/files/{TOKEN_SHARE}/2024-01/Empresas0.zip"
PASTA_TESTE = "tmp_teste_mascara"
NOME_ZIP = "Empresas0.zip"
CAMINHO_ZIP = os.path.join(PASTA_TESTE, NOME_ZIP)

colunas_empresa = [
    "cnpj_basico", "razao_social", "natureza_juridica",
    "qualificacao_responsavel", "capital_social", "porte_empresa",
    "ente_federativo_responsavel"
]

# 1. Download e Extração
os.makedirs(PASTA_TESTE, exist_ok=True)
if not os.path.exists(CAMINHO_ZIP):
    print("Baixando Empresas0.zip (Aprox. 150MB)...")
    with requests.get(URL_ZIP, stream=True) as r:
        r.raise_for_status()
        with open(CAMINHO_ZIP, 'wb') as f:
            for chunk in r.iter_content(chunk_size=1024*1024):
                f.write(chunk)

print("Extraindo...")
with zipfile.ZipFile(CAMINHO_ZIP, 'r') as zip_ref:
    zip_ref.extractall(PASTA_TESTE)
    nome_csv = zip_ref.namelist()[0]
    caminho_csv = os.path.join(PASTA_TESTE, nome_csv)

# 2. Leitura
print("Lendo as primeiras 500.000 linhas...")
df = pd.read_csv(caminho_csv, sep=';', header=None, names=colunas_empresa, 
                 encoding='iso-8859-1', dtype=str, nrows=500000)
df = df.fillna("")

# 3. Injeção de Casos Reais
casos_injetados = pd.DataFrame([
    {"cnpj_basico": "41781710", "razao_social": "MICHELLE CRISTINA R DE C TEIXEIRA-CPF10573183619", "porte_empresa": "01", "natureza_juridica": "2135"},
    {"cnpj_basico": "44320821", "razao_social": "DANIEL ALVES DAS CHAGAS-CPF065.459.296-90", "porte_empresa": "01", "natureza_juridica": "2135"},
    {"cnpj_basico": "44802356", "razao_social": "FLAVIO DE CARVALHO CAMAROTA05278413662", "porte_empresa": "01", "natureza_juridica": "2135"}
])
df = pd.concat([df, casos_injetados], ignore_index=True)

# 4. Filtro e Busca
print("\n--- ANÁLISE ANTES DA MÁSCARA ---")
df_mei = df[(df['porte_empresa'] == '01') & (df['natureza_juridica'] == '2135')].copy()

# A regex que captura o CPF e qualquer sujeira ao redor
regex_cpf_sujo = re.compile(r'(?i)[-.\s]*(?:CPF)?[-.\s]*(?<!\d)\d{3}\.?\d{3}\.?\d{3}-?\d{2}(?!\d)[-.\s]*')

mask_vazados = df_mei['razao_social'].str.contains(regex_cpf_sujo, regex=True)
df_vazados_antes = df_mei[mask_vazados]

print(f"Encontrados {len(df_vazados_antes)} MEIs com CPF exposto na amostra.")
for _, row in df_vazados_antes.tail(6).iterrows():
    print(f"CNPJ: {row['cnpj_basico']} | NOME: {row['razao_social']}")

# 5. Aplicação da Máscara (Padrão Atual da Receita com Proteção contra Duplicação)
print("\n--- APLICANDO O NOVO PADRÃO ---")
razoes_limpas = []
for razao, cnpj in zip(df_mei['razao_social'], df_mei['cnpj_basico']):
    razao_str = str(razao)
    cnpj_str = str(cnpj).zfill(8) # Garante que tem 8 dígitos
    
    # 1. Apaga o CPF, se existir
    nome_sem_cpf = regex_cpf_sujo.sub("", razao_str).strip()
    
    cnpj_formatado = f"{cnpj_str[:2]}.{cnpj_str[2:5]}.{cnpj_str[5:]}"
    
    # 2. Remove o CNPJ do início caso a Receita já o tenha colocado (evita o caso Avanilson)
    if nome_sem_cpf.startswith(cnpj_formatado):
        nome_sem_cpf = nome_sem_cpf[len(cnpj_formatado):]
    elif nome_sem_cpf.startswith(cnpj_str):
        nome_sem_cpf = nome_sem_cpf[len(cnpj_str):]
        
    # Limpa possíveis hífens ou espaços que tenham ficado no começo (ex: "- NOME")
    nome_sem_cpf = re.sub(r'^[-.\s]+', '', nome_sem_cpf).strip()
    
    # 3. Formata e insere no início de forma padronizada
    razoes_limpas.append(f"{cnpj_formatado} {nome_sem_cpf}")
    
df_mei['razao_social_mascarada'] = razoes_limpas

# 6. Resultados
print("\n--- RESULTADO DOS VAZADOS (ALTERADOS) ---")
df_vazados_depois = df_mei.loc[df_vazados_antes.index]
for _, row in df_vazados_depois.tail(6).iterrows():
    print(f"CNPJ: {row['cnpj_basico']} | NOME LIMPO: {row['razao_social_mascarada']}")

print("\n--- EXEMPLOS ANTIGOS/NORMAIS AGORA NO NOVO PADRÃO ---")
# Mostra que os registros puros antigos agora também ganharam o CNPJ formatado no início
df_nao_alterados = df_mei[~mask_vazados]
for _, row in df_nao_alterados.head(5).iterrows():
    print(f"CNPJ: {row['cnpj_basico']} | ORIGINAL: {row['razao_social']}")
    print(f"          -> NOME NOVO: {row['razao_social_mascarada']}")

# Limpeza do disco
os.remove(caminho_csv)
os.remove(CAMINHO_ZIP)
os.rmdir(PASTA_TESTE)
print("\nAmbiente de teste limpo.")